# Panel de balance mensual — pro-forma

Construye `data/interim/paneles_largos/panel_balance_mensual_proforma.parquet` a partir del panel principal, consolidando las fusiones/absorciones listadas en `data/external/crosswalks/fusiones.csv` con `tipo_evento = absorcion`.

La consolidación es **retroactiva**: la entidad absorbida se trata como si siempre hubiera sido parte del absorbente. Cada observación del absorbido se reasigna al absorbente y se suma al saldo correspondiente. La entidad absorbida desaparece del panel resultante.

Decisión metodológica: solo se aplican en el pro-forma las fusiones marcadas con `aplicar_en_proforma = TRUE` en `fusiones.csv`. El criterio es que la absorción caiga dentro del peak del shock CERA y pueda contaminar la señal durante la Etapa 1 del blanqueo.

Fusiones consolidadas en v1:
- `00259 (Banco BMA SAU) → 00285 (Banco Macro)` — fusión legal nov-2024, BMA deja de reportar en oct-2024. **Cae en el peak del shock, consolidamos**.

Fusiones NO consolidadas (se dejan as-is):
- `00150 (HSBC / GGAL) → 00007 (Banco Galicia)` — fusión legal jun-2025, GGAL deja de reportar en may-2025. Durante todo el peak CERA (sep-oct 2024) HSBC opera independiente. Disponible como robustez futura.

Ver `docs/notas/crosswalks.md §2` para la justificación detallada.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import PANELES, DIMENSIONES, EXTERNAL, REPO

import pandas as pd
import duckdb

PANEL_ORIG = PANELES / "panel_balance_mensual.parquet"
FUSIONES = EXTERNAL / "crosswalks/fusiones.csv"
OUT = PANELES / "panel_balance_mensual_proforma.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)

## Mapeo de absorbidos a absorbentes

Filtro las filas de `fusiones.csv` con `tipo_evento = 'absorcion'`, ambos códigos presentes, y `aplicar_en_proforma = TRUE`.

In [2]:
fusiones = pd.read_csv(FUSIONES, dtype=str)
absorciones = fusiones[
    (fusiones["tipo_evento"] == "absorcion")
    & fusiones["codigo_absorbido"].notna()
    & fusiones["codigo_absorbente"].notna()
    & (fusiones["codigo_absorbido"] != "")
    & (fusiones["codigo_absorbente"] != "")
    & (fusiones["aplicar_en_proforma"] == "TRUE")
].copy()

mapeo = absorciones[["codigo_absorbido", "codigo_absorbente", "fecha_legal", "nombre_absorbido", "nombre_absorbente"]]
mapeo

,codigo_absorbido,codigo_absorbente,fecha_legal,nombre_absorbido,nombre_absorbente
0,00259,00285,2024-11-19,Banco BMA SAU (ex Banco Itau Argentina),Banco Macro SA


## Construcción del panel pro-forma

La lógica en SQL: para cada fila del panel original, si el `codigo_entidad` aparece como `codigo_absorbido` en el mapeo, se reemplaza por el `codigo_absorbente`; después se agrega por `(codigo_entidad, yyyymm, codigo_cuenta)` sumando el saldo.

Esto funciona correctamente incluso cuando el absorbente y el absorbido reportaban en simultáneo en algún período: los saldos de ambos se suman en la fila consolidada.

In [3]:
q = f"""
    with mapeo as (
        select codigo_absorbido, codigo_absorbente
        from '{FUSIONES}'
        where tipo_evento = 'absorcion'
          and codigo_absorbido is not null and codigo_absorbente is not null
          and codigo_absorbido != '' and codigo_absorbente != ''
          and aplicar_en_proforma = 'TRUE'
    ),
    reasignado as (
        select coalesce(m.codigo_absorbente, p.codigo_entidad) as codigo_entidad,
               p.yyyymm,
               p.fecha,
               p.codigo_cuenta,
               p.saldo
        from '{PANEL_ORIG}' p
        left join mapeo m on p.codigo_entidad = m.codigo_absorbido
    )
    select codigo_entidad, yyyymm, fecha, codigo_cuenta, sum(saldo) as saldo
    from reasignado
    group by codigo_entidad, yyyymm, fecha, codigo_cuenta
    order by codigo_entidad, yyyymm, codigo_cuenta
"""
panel_pro = duckdb.sql(q).df()
print(f"Panel pro-forma: {len(panel_pro):,} filas")

Panel pro-forma: 1,373,060 filas


## Persistencia

In [4]:
panel_pro.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")

Escrito: data/interim/paneles_largos/panel_balance_mensual_proforma.parquet


## Validaciones

Chequeos específicos del pro-forma:
1. Los códigos absorbidos **no aparecen** en el panel resultante.
2. Los saldos totales del sistema deben coincidir entre panel original y pro-forma (consolidar no altera totales).
3. El absorbente tiene saldos más grandes que en el panel original (porque acumula al absorbido).

In [5]:
# 1. Absorbidos no deben aparecer
codigos_absorbidos = set(absorciones["codigo_absorbido"])
codigos_en_panel = set(panel_pro["codigo_entidad"].unique())
assert codigos_absorbidos.isdisjoint(codigos_en_panel), (
    f"Códigos absorbidos aparecen en el pro-forma: {codigos_absorbidos & codigos_en_panel}"
)

# 2. Totales del sistema deben conservarse
q_tot_orig = f"select yyyymm, sum(saldo) as saldo_total from '{PANEL_ORIG}' group by yyyymm"
q_tot_pro = f"select yyyymm, sum(saldo) as saldo_total from '{OUT}' group by yyyymm"
tot_orig = duckdb.sql(q_tot_orig).df().sort_values("yyyymm").reset_index(drop=True)
tot_pro = duckdb.sql(q_tot_pro).df().sort_values("yyyymm").reset_index(drop=True)

comp = tot_orig.merge(tot_pro, on="yyyymm", suffixes=("_orig", "_pro"))
comp["dif"] = comp["saldo_total_orig"] - comp["saldo_total_pro"]
assert comp["dif"].abs().max() < 1, f"Totales del sistema no coinciden: max diff = {comp['dif'].abs().max()}"

# 3. Spot check: Macro antes y después de la consolidacion
q = f"""
    select yyyymm,
           (select sum(saldo) from '{PANEL_ORIG}'
             where codigo_entidad = '00285' and yyyymm = a.yyyymm and codigo_cuenta like '315%') as macro_orig_dep_usd,
           (select sum(saldo) from '{OUT}'
             where codigo_entidad = '00285' and yyyymm = a.yyyymm and codigo_cuenta like '315%') as macro_pro_dep_usd
    from (select distinct yyyymm from '{PANEL_ORIG}' where yyyymm between 202401 and 202412) a
    order by yyyymm
"""
macro_comp = duckdb.sql(q).df()
macro_comp["delta_pro_minus_orig_miles_millones"] = (macro_comp["macro_pro_dep_usd"] - macro_comp["macro_orig_dep_usd"]) / 1e9
print("Depósitos USD de Macro: original vs pro-forma (aumentos positivos = saldos heredados de BMA)")
print(macro_comp.to_string(index=False))
print()
print("Validaciones OK")

Depósitos USD de Macro: original vs pro-forma (aumentos positivos = saldos heredados de BMA)
 yyyymm  macro_orig_dep_usd  macro_pro_dep_usd  delta_pro_minus_orig_miles_millones
 202401       -7.742044e+11      -1.001271e+12                          -227.066793
 202402       -8.026707e+11      -1.052704e+12                          -250.033408
 202403       -8.235884e+11      -1.085258e+12                          -261.669318
 202404       -8.874342e+11      -1.153409e+12                          -265.974840
 202405       -9.212929e+11      -1.204155e+12                          -282.862196
 202406       -9.629257e+11      -1.270537e+12                          -307.611006
 202407       -1.022186e+12      -1.358331e+12                          -336.144666
 202408       -1.175513e+12      -1.529436e+12                          -353.923276
 202409       -2.562598e+12      -2.893720e+12                          -331.121532
 202410       -2.729244e+12      -3.104587e+12                     

## Muestra: evolución del Macro pro-forma

In [6]:
q = f"""
    select yyyymm,
           abs(sum(case when codigo_cuenta like '315%' or codigo_cuenta like '316%' then saldo end)) / 1e12 as dep_usd_billones,
           abs(sum(case when codigo_cuenta in ('311793','312183','315794','316147') then saldo end)) / 1e12 as cera_billones
    from '{OUT}'
    where codigo_entidad = '00285' and yyyymm between 202401 and 202512
    group by yyyymm
    order by yyyymm
"""
duckdb.sql(q).df()

,yyyymm,dep_usd_billones,cera_billones
0,202401,1.001487,NaN
1,202402,1.053751,NaN
2,202403,1.086320,NaN
3,202404,1.153752,NaN
4,202405,1.204475,NaN
5,202406,1.270767,NaN
6,202407,1.358580,NaN
7,202408,1.529743,0.056169
8,202409,2.893937,1.445480
9,202410,3.104797,1.380585
